In [6]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [7]:
pip install -U langchain-text-splitters

In [8]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_7618/380848645.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [9]:
video_id = "8IU7YBgpQxg"

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=8IU7YBgpQxg! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have used to 

In [10]:
transcript_list

NameError: name 'transcript_list' is not defined

In [12]:
with open(r"/content/youtube_transcript.txt", 'r') as file:
    transcript = file.read()

## Step 1b - Indexing (Text Splitting)

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

In [14]:
spliter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

In [15]:
chunks = spliter.create_documents([transcript])

In [16]:
print(len(chunks))

88


## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [17]:

from langchain_community.embeddings import HuggingFaceEmbeddings

In [18]:
!pip install -q langchain-huggingface
from langchain_huggingface import HuggingFaceEmbeddings

In [19]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding,
)

In [22]:
vector_store.index_to_docstore_id

{0: 'b23aabd4-47d6-42e3-a9a9-6d696379db4d',
 1: 'faacab9d-1d9b-42cb-956c-586bdfb33f7e',
 2: 'beed4292-5192-40c6-b2a8-ea68f2ff4459',
 3: '776b3a10-ec33-47dc-bea2-d1d5275b92be',
 4: '9a93007e-c921-49bf-aebb-764b63b29687',
 5: 'cadc310b-7709-4e87-aff4-db85ba2a7d6e',
 6: '9b62a5a9-3d31-4d52-98f4-3892cde0afb6',
 7: 'fdc51b13-d76a-4b7a-b8fd-4c1ffc946aa2',
 8: 'df667e11-b617-4693-a397-83a99488714f',
 9: 'abf6019e-5ea4-4a69-a0c1-fe6b2b514a21',
 10: 'a43a3fa5-f5f0-430b-ad38-ae8f5169a5c8',
 11: '27f7a21a-3975-482e-9dc6-f8289bf4cacd',
 12: '1df7bb72-7ba3-4f4d-ae25-b6541b3a6cf9',
 13: '144c7dff-fdbc-4998-9b5b-48e5b3d6df86',
 14: '9a863b13-8168-4cfd-81db-c234e6dc33aa',
 15: 'b5f4ba6b-b09e-49cc-a412-417b55a7e280',
 16: '27eef209-7c62-459f-bf6a-b35fac2cfc6a',
 17: '37b85b6d-c0cc-450b-ba4c-8c55062e518d',
 18: '2bf56a34-2ecb-4d5a-b955-749e989db5e9',
 19: '361c4e26-2f40-4b43-953a-ccc53560e422',
 20: '587632c9-4003-40b1-8c0c-68ef6b692f6c',
 21: 'd24bfbb6-3cf4-48f0-a236-a6305969a8f4',
 22: 'edcfbb92-0a42-

## Step 2 - Retrieval

In [23]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [24]:
retriever.invoke('What is deepmind')

[Document(id='05b52ffc-7baa-4d6d-8edc-5a9618dfa57d', metadata={}, page_content="these little weapons these drones the these lering drones and even anti-personnel drones which take out one person or two people it's like a f flying hand grenade it will chase a soldier around and once it is like near enough it will explode it will delate and it's just a small determination like a single hand grenade but it can kill or MIM three or four people at at a given time so we are seeing this very new kind of warfare autonomous weapon systems that think on their own that don't always need inputs from the remote controlling person far away and they can take decisions on their own we are beginning to see the the the influence of AI in this you know very very rudimentary AI systems B put on a chip on a drone and this drone can go and kill um whoever it wants to kill on its own sometimes so we are seeing this happening a whole new level of warfare is coming up uh tanks could become obsolete if sufficie

## Step 3 - Augmentation

In [25]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [26]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [28]:
print(len(retrieved_docs))

4


In [29]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"have sufficient facile material nuclear facile material to perhaps construct a few cudim nuclear bombs and that is something that worries Israel a lot because Israel also is a nuclear weapons power and unstated nuclear weapons power but it's believed to have about let's say 90 or 100 nuclear warheads so there is all these Dynamics happening in the region there's also the Turkish angle north of Iran you have the nation of aeran which is a very strong Turkish Ally or proxy and they are trying to create this pan Turan Corridor which will essentially Wipe Out the nation of Armenia and create a whole turkey corridor from Turkey in the west all the way to turkistan and other turky nations in the East Central Asia region which is something that Iran would not like to see happen even the Americans would not like to see this happen even the Russians don't want this so there's a whole lot going on the Middle East is always a very complicated region but you have that and then what happens on\n\n

In [30]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

## Step 4 - Generation

In [32]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 4.0 MB/s eta 0:00:00


In [ ]:
# from google.colab import userdata
# GOOGLE_API_KEY = userdata.get('Gemini_API_KEY')

In [35]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\Rahul\OneDrive\Desktop\langchain\.env")   # Looks for .env in the current working directory

Gemini_API_KEY = os.getenv("Gemini_API_KEY")

my_token = os.getenv("HF_TOKEN")

In [36]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

In [37]:
answer = llm.invoke(final_prompt)
print(answer.content)

I don't know. The provided transcript discusses nuclear bombs, nuclear weapons, and nuclear warheads, but not nuclear fusion.


## Building a Chain

---



In [38]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [39]:
parser = StrOutputParser()

In [41]:
def format_docs(docs):
  context_texts = '/n/n'.join(doc.page_content for doc in docs)
  return context_texts

In [42]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [43]:
parallel_chain.invoke("What is Dermis ?")

{'context': "is now under the control of these Outsiders these infiltrators okay and uh over the past 10 years thousands of new Villages are visible in satellite images and maybe a million or so people must have infiltrated into into India from across the border now is it the government's fault no it is the fault of previous governments who did not ever fence the border and there was this open border regime that India and Burma had signed that the Border will remain open and people will have the right to move across the border for a certain amount of time and a certain amount of of kilometers so because of that these people have been able to infiltrate million close to a million people into India and the native Manipur is the people who speak the manipuri language which has been around for 2,000 years and who have always ruled Manipur they now occupy less than 7% of the of their ancestral territories so that's the deal that's what's happening in Manipur the the so-called cookies they h

In [44]:
chain = parallel_chain | prompt | llm | parser

In [45]:
chain.invoke('Can you summarize the video')

'The provided transcript covers several topics:\n\nIt details a recent Iranian attack on Israel involving cruise missiles, ballistic missiles, and drones. The attack was choreographed to have assets arrive simultaneously, allowing Israel to detect and neutralize most of them with assistance from Jordan and Saudi Arabia, and its Iron Dome system, resulting in limited damage.\n\nThe transcript also discusses the unexpected nature of the Hamas War and identifies several global "flash points" including West Asia, India-Pakistan, India-China, South China Sea/Taiwan, East Asia (North/South Korea), and the Ukraine conflict, noting that miscalculations in these regions can lead to problems.\n\nAnother issue highlighted is the challenge of foreign infiltration into India, the difficulty of border management, and the limited utility of drones in thick forest cover.\n\nFrom a broader geopolitical perspective, the speaker describes the current year as a "pivotal year," with potential for escalatio

#FINAL RAG

In [47]:
# !pip install -q youtube-transcript-api langchain-community langchain-openai \
#                faiss-cpu tiktoken python-dotenv




# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain.schema.output_parser import StrOutputParser
# from langchain.schema.runnable import RunnableParallel, RunnablePassthrough, RunnableLambda
# from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
# # from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.vectorstores import FAISS
# from langchain_core.prompts import PromptTemplate

# video_id = input("Enter the video id: ")
# if not video_id:
#   video_id = "8IU7YBgpQxg"

# try:
#     fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
#     transcript_list = fetched_transcript.to_raw_data()
#     transcript = " ".join(chunk["text"] for chunk in transcript_list)
#     print(transcript)

# except TranscriptsDisabled:
#     print("No captions available for this video.")
# except Exception as e:
#     print(f"An error occurred: {e}")


# from google.colab import userdata
# GOOGLE_API_KEY = userdata.get('Gemini_API_KEY')



# parser = StrOutputParser()
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     google_api_key=GOOGLE_API_KEY
# )


# spliter = RecursiveCharacterTextSplitter(
#     chunk_size = 1000,
#     chunk_overlap = 200
# )

# chunks = spliter.create_documents([transcript])

# embedding = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )

# # embeddings = embedding.embed_documents(documents)

# vector_store = FAISS.from_documents(
#     documents=chunks,
#     embedding=embedding,
# )

# retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# prompt = PromptTemplate(
#     template="""
#       You are a helpful assistant.
#       Answer ONLY from the provided transcript context.
#       If the context is insufficient, just say you don't know.

#       {context}
#       Question: {question}
#     """,
#     input_variables = ['context', 'question']
# )

# def format_docs(docs):
#   context_texts = '/n/n'.join(doc.page_content for doc in docs)
#   return context_texts

#   parallel_chain = RunnableParallel({
#     'context' : retriever | RunnableLambda(format_docs),
#     'question' : RunnablePassthrough()
# })

# chain = parallel_chain | prompt | llm | parser

# question = input("Enter your question: ")
# answer = chain.invoke(question)
# print(answer)


Enter the video id: 
An error occurred: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=! This is most likely caused by:

The video is no longer available

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!


TypeError: RecursiveCharacterTextSplitter.from_language() missing 1 required positional argument: 'language'

In [ ]:
# Added missing packages: langchain-google-genai, langchain-huggingface, sentence-transformers
!pip install -q youtube-transcript-api langchain-community langchain-google-genai langchain-huggingface faiss-cpu tiktoken sentence-transformers

import sys
# from google.colab import userdata
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Fetch API Key securely
# try:
#     GOOGLE_API_KEY = userdata.get('Gemini_API_KEY')
# except Exception as e:
#     print("Error: Could not retrieve Gemini_API_KEY from Colab secrets.")
#     sys.exit()
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\Rahul\OneDrive\Desktop\langchain\.env")   # Looks for .env in the current working directory

Gemini_API_KEY = os.getenv("Gemini_API_KEY")

my_token = os.getenv("HF_TOKEN")

# 2. Get Video ID
video_id = input("Enter the video id (leave blank for default): ")
if not video_id:
    video_id = "8IU7YBgpQxg"

# 3. Fetch Transcript
try:
    # Corrected YouTubeTranscriptApi usage
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=['en'])
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print("\nTranscript fetched successfully! Processing...\n")
except TranscriptsDisabled:
    print("No captions available for this video.")
    sys.exit()
except Exception as e:
    print(f"An error occurred: {e}")
    sys.exit()

# 4. Initialize LLM
parser = StrOutputParser()
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

# 5. Split Transcript into Chunks
spliter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = spliter.create_documents([transcript])

# 6. Create Embeddings and Vector Store
# Added proper HuggingFace import and initialization
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding,
)

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 7. Define Prompt
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=['context', 'question']
)

# 8. Format Docs and Setup Chain
def format_docs(docs):
    # Corrected forward slashes (/n/n) to backslashes (\n\n)
    return '\n\n'.join(doc.page_content for doc in docs)

# Corrected the indentation error here
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})



chain = parallel_chain | prompt | llm | parser



# 9. Ask Question and Invoke
question = input("Enter your question: ")
answer = chain.invoke(question)



print("\n--- Answer ---")
print(answer)